# Figure 4. Genetic and molecular architecture of the TMEM106B locus

## Setup

In [ ]:
suppressMessages({library(data.table); library(tidyverse); library(ggplot2); library(patchwork); library(colocboost)})
source('data/colocboost_plot.R')   # local colocboost_plot() with annotation / bottom-track support
# shared palette: protein domains, high-mannose teal, hybrid/complex blue, unadjusted
col_domain <- c(N = '#2E6DB4', TM = '#D49A1A', C = '#4A9D4E'); col_hm <- '#66C2A5'; col_hl <- '#2E6DB4'; col_lead <- '#d7301f'; col_ns <- '#9aa0a6'
tag_theme <- theme(plot.tag = element_text(size = 27, face = 'bold', hjust = 0, vjust = 1), plot.tag.position = c(0, 1))
theme_panel <- function(bs = 14) theme_bw(base_size = bs) + theme(panel.grid = element_blank(), panel.border = element_rect(linewidth = 1.0),
  axis.title = element_text(size = bs + 1, face = 'bold'), axis.text = element_text(colour = 'black'), plot.margin = margin(4, 9, 4, 9))
glycan_sym <- function(df, u, hexnac_col = 'hexnac', hex_col = 'hex', size = 4, txt = 4.2) {
  df <- as.data.frame(df)
  list(geom_text(data = df, aes(x = x - 2.1 * u, y = y, label = paste0(.data[[hexnac_col]], '\u00d7')), inherit.aes = FALSE, size = txt),
       geom_point(data = df, aes(x = x - 1.1 * u, y = y), inherit.aes = FALSE, shape = 22, fill = '#2F7ED8', colour = '#1F4E79', size = size, stroke = 0.6),
       geom_text(data = df, aes(x = x + 0.5 * u, y = y, label = paste0(.data[[hex_col]], '\u00d7')), inherit.aes = FALSE, size = txt),
       geom_point(data = df, aes(x = x + 1.5 * u, y = y), inherit.aes = FALSE, shape = 21, fill = '#68C83E', colour = '#2E6B25', size = size, stroke = 0.6))
}

## Panel a: schematic image

In [ ]:
# Panel a: TMEM106B domain / glycosite schematic
a_png <- 'data/TMEM106B_Glycosites.png'
a_arr <- png::readPNG(a_png); IMG_H <- dim(a_arr)[1]; IMG_W <- dim(a_arr)[2]
isc <- function(r, g, b, tol = 0.09) abs(a_arr[,,1] - r/255) < tol & abs(a_arr[,,2] - g/255) < tol & abs(a_arr[,,3] - b/255) < tol & a_arr[,,4] > 0.9
m_blue <- isc(59, 109, 184); m_green <- isc(66, 150, 73)
bar_rows <- which(rowSums(m_blue) > 0.5 * max(rowSums(m_blue))); bar_row <- median(bar_rows); bar_bottom <- max(bar_rows)
bx <- range(which(m_blue[bar_row, ])); gx <- range(which(m_green[bar_row, ]))   # N-terminal 1-96, luminal 118-274
c(blue = bx, green = gx, bar_bottom = bar_bottom, IMG_W = IMG_W, IMG_H = IMG_H)

## Heatmap (below a): detection by genotype

In [ ]:
# Panel a (bottom): landscape heatmap, fraction of donors with a quantified T185 / S185
hm <- readRDS('data/Figure_4a_heatmap.rds')[glycan != 'Hex8']
hm[, `:=`(sx = as.numeric(factor(signal, c('T185', 'S185', 'Any'))), gy = c('G/G' = 1, 'C/G' = 2, 'C/C' = 3)[genotype], label = sprintf('%d/%d\n%.0f%%', n, N, 100 * fraction),
          hex = as.integer(sub('Hex', '', glycan)), hexnac = 2L)]
hm_lab <- unique(hm[, .(glycan, hex, hexnac)])[, `:=`(x = 2, y = 3.95)]
f_a_heat <- ggplot(hm, aes(sx, gy)) +
  geom_tile(aes(fill = fraction), colour = 'white', linewidth = 0.8) +
  geom_text(aes(label = label, colour = fraction > 0.55), size = 4.4, lineheight = 0.9) +
  glycan_sym(hm_lab, u = 0.25, size = 5.5, txt = 5.6) +
  scale_colour_manual(values = c('FALSE' = '#203642', 'TRUE' = 'white'), guide = 'none') +
  scale_fill_gradient(low = '#F0F3F5', high = col_hl, limits = c(0, 1), guide = 'none') +
  scale_x_continuous(breaks = 1:3, labels = c('T185', 'S185', 'Any'), expand = c(0, 0.02)) +
  scale_y_continuous(breaks = 1:3, labels = c('G/G', 'C/G', 'C/C'), expand = c(0, 0)) +
  coord_cartesian(ylim = c(0.5, 3.5), clip = 'off') +   # ylim via coord so the glycan symbols above the frame are not dropped
  facet_grid(. ~ glycan) + labs(x = NULL, y = 'rs3173615') + tag_theme +
  theme_panel(13) + theme(strip.background = element_blank(), strip.text = element_blank(), panel.spacing.x = unit(5, 'pt'),
                          axis.text.y = element_text(size = 14.5), axis.text.x = element_text(size = 14.5), axis.title.y = element_text(size = 16, face = 'bold'), plot.margin = margin(40, 9, 4, 9))
f_a_heat

## Panel b: peptide cis-pQTL on the schematic axis

In [ ]:
# Panel b: per-peptide cis-pQTL effect of rs3173615
qres <- as.data.table(readRDS('data/Figure_4b_pqtl_effects.rds'))
pep_start <- c(EDAYDGVTSENMR = 15, NGDVSQFPYVEFTGR = 42, DSVTCPTCQGTGR = 57, GQENQLVALIPYSDQR = 73, SAYVSYDVQK = 130, YQYVDCGR = 248)  # UniProt Q9NUM4
qres[, lab := c(EDAYDGVTSENMR = 'N[1]', NGDVSQFPYVEFTGR = 'N[2]', DSVTCPTCQGTGR = 'N[3]', GQENQLVALIPYSDQR = 'N[4]', SAYVSYDVQK = 'C[core]', YQYVDCGR = 'C[boundary]')[as.character(peptide)]]
qres[, start := pep_start[peptide]][, end := start + nchar(as.character(peptide)) - 1][, lead := peptide == 'SAYVSYDVQK']
res2px <- function(r) ifelse(r <= 96, bx[1] + (r - 1) * diff(bx) / 95, gx[1] + (r - 118) * diff(gx) / 156)
qres[, `:=`(xm = res2px((start + end) / 2), lo = beta - 1.96 * se, hi = beta + 1.96 * se, ptcol = fifelse(lead, col_lead, col_ns))]
W_col <- 11; H_col <- 9.52; L_in <- 1.1; IMG_IN_W <- W_col - L_in; s <- IMG_IN_W / IMG_W
IMG_IN_H <- IMG_IN_W * IMG_H / IMG_W; HEAT_IN <- 3.3; PEP_IN <- 3.07
top_img <- 0; top_heat <- IMG_IN_H + 0.03; top_pep <- top_heat + HEAT_IN + 0.55      # 0.55 in of free arrow between heatmap and peptide plot
ytop <- -0.1 / s; EH <- 2.1 / s; ybot <- ytop - EH
bmax <- 0.52; bmin <- -0.88; ymap <- function(b) ytop - (bmax - b) / (bmax - bmin) * EH
xt <- c(1, 96, 183, 274); XL <- bx[1]; XR <- gx[2]
f_pep <- ggplot() +
  annotate('rect', xmin = XL, xmax = XR, ymin = ybot, ymax = ytop, fill = NA, colour = 'black', linewidth = 0.8) +
  annotate('segment', x = XL, xend = XR, y = ymap(0), yend = ymap(0), colour = 'grey60', linewidth = 0.3) +
  geom_errorbar(data = qres, aes(x = xm, ymin = ymap(lo), ymax = ymap(hi)), width = 28, colour = qres$ptcol, linewidth = 0.5) +
  geom_point(data = qres, aes(xm, ymap(beta)), colour = qres$ptcol, size = 3.2) +   # all peptides filled; C_core red, others grey
  annotate('segment', x = min(res2px(15)), xend = res2px(73 + 15), y = ymap(0.31), yend = ymap(0.31), colour = col_ns, linewidth = 0.5) +
  annotate('text', x = (res2px(15) + res2px(73 + 15)) / 2, y = ymap(0.40), label = 'N-terminal peptides (n.s.)', size = 5.4, colour = col_ns, hjust = 0.5) +
  annotate('text', x = qres[lead == TRUE, xm], y = ymap(qres[lead == TRUE, lo]) - 0.06 / s, label = 'bold(C[core]~"***")', parse = TRUE, size = 5.2, colour = col_lead, vjust = 1) +
  annotate('text', x = qres[peptide == 'YQYVDCGR', xm] + 0.1 / s, y = ymap(qres[peptide == 'YQYVDCGR', lo]) - 0.1 / s, label = 'atop(C[boundary], "(non-monotonic)")', parse = TRUE, size = 5.2, colour = col_ns, vjust = 1, hjust = 1) +
  geom_text(data = qres[!lead & peptide != 'YQYVDCGR'], aes(xm, ymap(lo) - 0.06 / s, label = lab), parse = TRUE, vjust = 1, size = 5.2, colour = col_ns) +   # N[1] to N[4] (plotmath subscripts) below each interval
  annotate('segment', x = XL, xend = XL - 0.06 / s, y = ymap(c(-0.8, -0.4, 0)), yend = ymap(c(-0.8, -0.4, 0))) +
  annotate('text', x = XL - 0.09 / s, y = ymap(c(-0.8, -0.4, 0)), label = c('-0.8', '-0.4', '0.0'), hjust = 1, size = 5.0) +
  annotate('text', x = XL - 0.62 / s, y = (ytop + ybot) / 2, label = 'cis-pQTL effect (\u03b2)\nSD per G allele', angle = 90, size = 5.0, lineheight = 0.9, fontface = 'bold') +
  annotate('segment', x = res2px(xt), xend = res2px(xt), y = ybot, yend = ybot - 0.06 / s) +
  annotate('text', x = res2px(xt), y = ybot - 0.1 / s, label = xt, vjust = 1, size = 5.0) +
  annotate('text', x = (XL + XR) / 2, y = ybot - 0.42 / s, label = 'TMEM106B residue', vjust = 1, size = 5.4, fontface = 'bold') +
  coord_fixed(xlim = c(-L_in / s, IMG_IN_W / s), ylim = c(-PEP_IN / s, 0), expand = FALSE) + theme_void() + theme(plot.margin = margin(0, 0, 0, 0))
f_pep

## Panel c: colocboost locus zoom

In [ ]:
# Panel c: colocboost locus zoom
fc <- readRDS('data/Figure_4c_locuszoom.rds')
stopifnot(packageVersion('colocboost') >= '1.0.10')
fc$cb_res <- get_robust_colocalization(fc$cb_res, cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 0.0001)
stopifnot(length(strsplit(fc$cb_res$cos_summary$colocalized_variables, '; ?')[[1]]) < 1000)   # guard against CoS expansion
glyph <- function(count, pch, fill, border) list(count = count, pch = pch, fill = fill, border = border)
glycan_annotation <- function(hexnac, hex) list(line1 = 'TMEM106B N183', glycan = list(glyph(hexnac, 22, '#2F7ED8', '#1F4E79'), glyph(hex, 21, '#68C83E', '#2E6B25')), component_gap = 0.099, symbol_offset = 0.055, glycan_y_offset = 0.115)
outcome_annotations <- list(list(line1 = 'TMEM106B', line2 = 'Protein'), glycan_annotation(2, 5), glycan_annotation(2, 6), glycan_annotation(2, 7),
  list(line1 = 'TMEM106B N-terminal', line2 = quote(paste('peptides (N'[mean], ')'))), list(line1 = 'TMEM106B C-terminal', line2 = quote(paste('peptide (C'[core], ')'))), list(line1 = 'ADRD', line2 = 'GWAS'))
options(colocboost_plot.uncoloc_cos_label = 'Colocalization not supported')   # legend wording for CoS variants not colocalized with an outcome (Man7, N_mean)
ord <- c(7, 6, 5, 2, 3, 4)   # ADRD, C_core (SAYVSYDVQK), N-terminal peptides (N_mean), Man5-Man7 (TMEM106B protein / pQTL not shown for now)
highlight_pos <- c(12229185, 12229791)
draw_gene_track <- function(x_range, tick_at, column, axis_size) {
  plot.new(); plot.window(xlim = x_range, ylim = c(-0.35, 0.35), xaxs = 'r', yaxs = 'i')
  with(fc$intron_df, arrows(start, 0, end, 0, length = 0.045, angle = 25, code = 2, lwd = 0.7))
  with(fc$ex_df, rect(start, -0.18, end, 0.18, col = 'steelblue', border = NA))
  axis(1, at = tick_at, labels = paste0(formatC(tick_at / 1e6, format = 'f', digits = 2), 'M'), cex.axis = axis_size)
  op <- par(xpd = NA); mtext('TMEM106B', side = 1, line = 2.5, cex = 1.25, font = 2); par(op); box(lwd = 0.8)   # xpd = NA; label raised from line 3.1 to 2.5 so it is not cut at the bottom edge of the c inset, tick labels still clear
}
draw_figure_4c <- function() colocboost_plot(fc$cb_res, grange = fc$grange[1]:fc$grange[2], variant_coord = TRUE, xlab = '',
  outcome_idx = ord, outcome_annotations = outcome_annotations, bottom_track = draw_gene_track, compact_x_axes = TRUE, track_height = 0.35,
  panel_top_margin = 0.5, outer_top_margin = 0, panel_axis_size = 1.7, lab_style = c(1.7, 1), annotation_scale = 1.1,
  add_highlight = TRUE, add_highlight_idx = highlight_pos, add_highlight_style = 'lead_coral_coding_hollow', add_highlight_uncoloc_idx = c(4, 5),   # Man7 and N-terminal peptides do not colocalize: hollow ring + x, no filled dot
  outcome_legend_size = 1.3, show_cos_to_uncoloc = TRUE, cos_legend_pos = c(0.05, 0.4, 0.8, 0.5),   # 3rd value: gap between the x symbol and 'Colocalization not supported' (default 0.1 was too tight)
 
  panel_left_margin_by_col = c(5, 3.4), hide_ylab_cols = 2, panel_column_widths = c(1, 0.936),   # right column: no -log10(P) label, tighter margin, equal plot widths
  panel_y_ticks = list(NULL, NULL, NULL, NULL, NULL, c(0, 1, 2, 3)))   # Man7 panel is short; avoid overlapping ticks
f_c <- patchwork::wrap_elements(full = gridGraphics::echoGrob(draw_figure_4c, prefix = 'figure4c')) + labs(tag = 'c') + tag_theme + theme(plot.tag.position = c(-0.02, 1))   # vector grob; tag shifted left of the plot

## Panel d: glycoQTL effect, unadjusted vs adjusted

In [ ]:
# Colours follow the Figure 5 palette
fd <- readRDS('data/Figure_4d_qtl_adjusted.rds'); qd <- as.data.table(fd$q)[adjustment %in% c('unadjusted', 'adjusted') & gp %in% c('8489', '8490', '8491')]
qd[, `:=`(form = factor(c('8489' = 'Man5', '8490' = 'Man6', '8491' = 'Man7')[gp], c('Man5', 'Man6', 'Man7')),
          adjustment = factor(adjustment, c('adjusted', 'unadjusted'), c('Adjusted for C_core', 'Unadjusted')))]
x_lab <- 0.12; u <- 0.45
lab <- data.table(form = factor(c('Man5', 'Man6', 'Man7'), c('Man5', 'Man6', 'Man7')), hex = c(5L, 6L, 7L), y = 1.5)
f_d <- ggplot(qd, aes(beta, adjustment, colour = adjustment)) +
  geom_vline(xintercept = 0, linetype = 'dashed', colour = 'grey50', linewidth = 0.3) +
  geom_errorbar(aes(xmin = ci_lo, xmax = ci_hi), width = 0.25, linewidth = 0.9, orientation = 'y') +
  geom_point(size = 4.0) +
  geom_text(aes(x = ci_lo, label = paste0('P = ', ifelse(p >= 0.01, sprintf('%.3f', p), format(p, digits = 2)))), vjust = -1, hjust = 1, size = 5.4, show.legend = FALSE) +
  geom_text(data = lab, aes(x = x_lab, y = y + 1.88 * u, label = '2×'), inherit.aes = FALSE, angle = -90, size = 6.5) +
  geom_point(data = lab, aes(x = x_lab, y = y + 1.2 * u), inherit.aes = FALSE, shape = 22, fill = '#2F7ED8', colour = '#1F4E79', size = 7, stroke = 0.6) +
  geom_text(data = lab, aes(x = x_lab, y = y - 0.3 * u, label = paste0(hex, '×')), inherit.aes = FALSE, angle = -90, size = 6.5) +
  geom_point(data = lab, aes(x = x_lab, y = y - 0.98 * u), inherit.aes = FALSE, shape = 21, fill = '#68C83E', colour = '#2E6B25', size = 7, stroke = 0.6) +
  facet_grid(form ~ .) +
  scale_colour_manual(values = c('Unadjusted' = '#1f78b4', 'Adjusted for C_core' = '#4D4D4D'), name = NULL, breaks = c('Unadjusted', 'Adjusted for C_core'), labels = expression('Unadjusted', 'Adjusted for C'[core])) +   # legend label with plotmath subscript
  coord_cartesian(xlim = c(-0.72, 0.04), clip = 'off') +
  labs(x = 'cis-glycoQTL effect of rs3173615 (95% CI)', y = NULL, tag = 'd') +
  theme_panel(16) + theme(panel.border = element_rect(linewidth = 0.7)) + tag_theme + theme(legend.position = 'top', legend.justification = 'left', legend.margin = margin(0, 0, 0, 0), legend.box.margin = margin(0, 0, 0, 55), legend.text = element_text(size = 18), legend.key.size = unit(8, 'mm'),
        axis.text.y = element_blank(), axis.text.x = element_text(size = 18), axis.title.x = element_text(size = 16, face = 'bold'), axis.ticks.y = element_blank(), strip.text = element_blank(), strip.background = element_blank(), panel.spacing.y = unit(8, 'pt'),
        plot.margin = margin(4, 60, 4, 45))
f_d

## Panel e: S185 / T185 ratio

In [ ]:
# Colour: sequential green gradient
hex_cols <- setNames(c('#B5E3A1', '#68C83E', '#2E6B25'), paste0('HexNAc(2)Hex(', 5:7, ')'))
fe <- readRDS('data/Figure_4e_ratio.rds'); de <- fe$d[is.finite(fe$d$log2_ratio), ]; te <- fe$tests; se_ <- fe$summary
fmt <- function(p) format.pval(p, digits = 3, eps = 1e-99)
te$x <- match(te$first, se_$glycan); te$xend <- match(te$second, se_$glycan)
top <- max(de$log2_ratio); te$y <- top + c(0.4, 1.9, 1.15); te$label <- paste0('P = ', fmt(te$p), ' (paired n = ', te$n, ')')
f_e <- ggplot(de, aes(glycan, log2_ratio)) +
  geom_hline(yintercept = 0, linetype = 2, colour = 'grey50') +
  annotate('text', x = 2.5, y = 0, label = 'Equal\nabundance', colour = 'grey40', size = 4.8, hjust = 0.5, vjust = -0.15, lineheight = 0.85) +
  geom_boxplot(aes(fill = glycan), width = 0.5, outlier.shape = NA, alpha = 0.55, colour = 'black') +
  geom_point(position = position_jitter(width = 0.12, seed = 185), aes(colour = glycan), size = 2.6, alpha = 0.7) +
  scale_fill_manual(values = hex_cols) + scale_colour_manual(values = hex_cols) +
  geom_segment(data = te, aes(x = x, xend = xend, y = y, yend = y), inherit.aes = FALSE) +
  geom_segment(data = te, aes(x = x, xend = x, y = y, yend = y - 0.12), inherit.aes = FALSE) +
  geom_segment(data = te, aes(x = xend, xend = xend, y = y, yend = y - 0.12), inherit.aes = FALSE) +
  geom_text(data = te, aes(x = (x + xend)/2, y = y + 0.07, label = label), inherit.aes = FALSE, vjust = 0, size = 5.4) +
  scale_x_discrete(labels = NULL) +
  annotate('text', x = 1:3, y = min(de$log2_ratio) - 1.3, label = paste0('n = ', se_$n), size = 6.2) +
  glycan_sym(data.frame(x = 1:3, y = min(de$log2_ratio) - 0.8, hexnac = 2L, hex = 5:7), u = 0.13, size = 7.5, txt = 6.6) +
  coord_cartesian(ylim = range(c(de$log2_ratio, te$y + 0.55)), clip = 'off') +
  labs(x = NULL, y = 'log\u2082(S185/T185)', tag = 'e') + theme_panel(21) + theme(legend.position = 'none', panel.border = element_rect(linewidth = 0.7)) + tag_theme + theme(plot.tag.position = c(0, 1.077), plot.margin = margin(39, 22, 45.5, 16), axis.ticks.x = element_blank(), axis.title.y = element_text(size = 16, face = 'bold'), axis.text.y = element_text(size = 18))
f_e

## Panel f: mediation diagrams, forward (top) and switched roles (bottom), images as provided

In [ ]:
# Panels f and g: mediation diagrams, images as provided
med_panel <- function(file, tag, sc = 0.9) { im <- png::readPNG(file); hh <- dim(im)[1]; ww <- dim(im)[2]; px <- ww * (1 / sc - 1) / 2; py <- hh * (1 / sc - 1) / 2
  ggplot() + annotation_raster(im, 0, ww, 0, hh) + coord_fixed(xlim = c(-px, ww + px), ylim = c(-py, hh + py), expand = FALSE) +
    labs(tag = tag) + theme_void() + tag_theme + theme(plot.tag.position = c(0, 0.99), plot.margin = margin(2, 6, 2, 6)) }
f_f <- med_panel('data/mediation.png', 'f'); f_g <- med_panel('data/mediation_rev.png', NULL)   # no tag: both diagrams belong to panel f
f_f

## Combine (a / heatmap / c + arrow, absolute layout)

In [ ]:
# Panel a (schematic, with the detection heatmap below it) over b (peptide pQTL)
npc_y <- function(top_in) 1 - top_in / H_col
HEAT_A <- 0.95   # inches from the heatmap plot's left edge to its panel frame (y title + labels + margin)
x_sayv <- res2px(mean(c(pep_start[['SAYVSYDVQK']], pep_start[['SAYVSYDVQK']] + 9)))      # residues 130-139
x_arrow <- (L_in + x_sayv * s) / W_col
m_box <- isc(200, 98, 98, 0.12); m_box[1:(bar_bottom + 2), ] <- FALSE; m_box[, abs(seq_len(IMG_W) - x_sayv) > 0.03 * IMG_W] <- FALSE   # box rows under SAYV
y_arrow0 <- (max(which(rowSums(m_box) > 0)) + 8) / IMG_H * IMG_IN_H; y_arrow1 <- top_pep + (-ymap(qres[lead == TRUE, hi])) * s - 0.32   # stops well above the estimate
f_a <- cowplot::ggdraw(xlim = c(0, 1), ylim = c(0, 1)) +
  cowplot::draw_grob(grid::rasterGrob(png::readPNG(a_png), interpolate = TRUE), x = L_in / W_col, y = npc_y(IMG_IN_H), width = IMG_IN_W / W_col, height = IMG_IN_H / H_col, hjust = 0, vjust = 0) +
  cowplot::draw_line(x = c(x_arrow, x_arrow), y = c(npc_y(y_arrow0), npc_y(y_arrow1)), colour = col_lead, size = 0.7, linetype = 'dashed',
                     arrow = grid::arrow(length = unit(3.2, 'mm'), type = 'closed')) +
  cowplot::draw_plot(f_a_heat, x = (L_in + XL * s - HEAT_A) / W_col, y = npc_y(top_heat + HEAT_IN), width = (HEAT_A + (XR - XL) * s + 0.125) / W_col, height = HEAT_IN / H_col) +
  cowplot::draw_plot(f_pep, x = 0, y = npc_y(top_pep + PEP_IN), width = 1, height = PEP_IN / H_col) +
  cowplot::draw_label('a', x = 0.004, y = npc_y(0.05), hjust = 0, vjust = 1, fontface = 'bold', size = 27) +
  cowplot::draw_label('b', x = 0.004, y = npc_y(top_pep + 0.02), hjust = 0, vjust = 1, fontface = 'bold', size = 27)
FW <- 22; BOT <- 9.55; FH <- BOT + 7.6   # a-c unchanged; the bottom row (d-g) is taller so larger fonts fit
ins <- function(p, x0, x1, top, bot) inset_element(p, left = x0 / FW, right = x1 / FW, top = 1 - top / FH, bottom = 1 - bot / FH, align_to = 'full', clip = FALSE)
combined_fig <- ggplot() + theme_void() + ins(f_a, 0, 11, 0, H_col) + ins(f_c, 11.5, 22, 0, H_col) +
  ins(f_d, 0, 7, BOT, FH) + ins(f_e, 7, 15, BOT, FH) + ins(f_f, 15, 22, BOT, BOT + 3.8) + ins(f_g, 15, 22, BOT + 3.8, FH)
options(repr.plot.width = 22, repr.plot.height = FH); combined_fig

## Save

In [ ]:
ggsave('Figure4.pdf', combined_fig, device = grDevices::cairo_pdf, width = FW, height = FH, units = 'in', limitsize = FALSE, bg = 'white')
ggsave('Figure4.png', combined_fig, device = ragg::agg_png, width = FW, height = FH, units = 'in', dpi = 300, limitsize = FALSE, bg = 'white')   # same figure as PNG, 300 dpi